In [1]:
# load libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from lifelines.utils import concordance_index

import pyhere as here
import random
import copy

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Data Loading and Initial Processing

In [2]:
# Load clinical metadata
clinical = pd.read_csv(here.here("data", "processed", "clinical_data.csv"))
# Load expression data
expression_data = pd.read_csv(here.here("data", "processed", "expression_data.csv"))
# Mutation binary data
mutation_binary = pd.read_csv(here.here("data", "processed", "mutation_binary_data.csv"))
# Mutation classified data
mutation_classified = pd.read_csv(here.here("data", "processed", "mutation_classified_data.csv"))

pd.set_option('display.max_columns', None)

print("Data shapes:")
print(f"  Clinical: {clinical.shape}")
print(f"  Expression: {expression_data.shape}")
print(f"  Mutation Binary: {mutation_binary.shape}")
print(f"  Mutation Classified: {mutation_classified.shape}")

Data shapes:
  Clinical: (1904, 31)
  Expression: (1904, 489)
  Mutation Binary: (1904, 173)
  Mutation Classified: (1904, 173)


In [3]:
# Prepare clinical data
df = clinical.copy()
time_col = "overall_survival_months"
event_col = "death_from_cancer"

# Define outcome variables
df["event"] = (df[event_col] == "Died of Disease").astype(int)
df["time"] = df[time_col]

print(f"Event distribution:\n{df['event'].value_counts()}")

# Group rare cancer types into "Other" category
rare_threshold = 50

value_counts = df["cancer_type_detailed"].value_counts()
rare_classes = value_counts[value_counts < rare_threshold].index
df["cancer_type_detailed_clean"] = df["cancer_type_detailed"].replace(rare_classes, "Other")

# Do same for histologic subtype
value_counts = df["tumor_other_histologic_subtype"].value_counts()
rare_classes = value_counts[value_counts < rare_threshold].index
df["tumor_other_histologic_subtype_clean"] = df["tumor_other_histologic_subtype"].replace(rare_classes, "Other")

# Drop identifiers, outcome columns, and pre-modified columns
drop_cols = [
    "patient_id", "cancer_type", "cohort", "cancer_type_detailed",
    "er_status_measured_by_ihc", "her2_status_measured_by_snp6",
    "tumor_other_histologic_subtype", "integrative_cluster", "oncotree_code",
    "radio_therapy", "chemotherapy", "type_of_breast_surgery", "hormone_therapy",
    "overall_survival", "nottingham_prognostic_index", "3-gene_classifier_subtype",
    "tumor_other_histologic_subtype_clean", time_col, event_col
]

X_clinical = df.drop(columns=drop_cols + ["event", "time"])
y_time = df["time"].values
y_event = df["event"].values

# Identify feature types
numeric_cols = X_clinical.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_clinical.select_dtypes(include=["object", "category"]).columns.tolist()

# Fill missing categorical data
X_clinical[categorical_cols] = X_clinical[categorical_cols].fillna("Unknown")

print(f"\nClinical features:")
print(f"  Numeric: {len(numeric_cols)}")
print(f"  Categorical: {len(categorical_cols)}")

Event distribution:
event
0    1282
1     622
Name: count, dtype: int64

Clinical features:
  Numeric: 5
  Categorical: 9


In [4]:
# One-hot encode categorical columns
X_clinical_encoded = pd.get_dummies(X_clinical, columns=categorical_cols, drop_first=True)

# Scale numeric features
scaler_clinical = StandardScaler()
X_clinical_encoded[numeric_cols] = scaler_clinical.fit_transform(X_clinical_encoded[numeric_cols])

print(f"Clinical features after encoding: {X_clinical_encoded.shape}")

Clinical features after encoding: (1904, 29)


In [5]:
# Process mutation data: create 3 complementary feature sets

# 1. Binary mutation matrix (0/1)
mut_binary_vals = mutation_binary.values.astype(np.float32)

# 2. Severity-weighted matrix
severity_map = {
    'WT':         0.0,
    'Missense':   0.5,
    'Other':      0.3,
    'Frameshift': 1.0,
    'Nonsense':   1.0,
    'Deletion':   0.8,
    'Insertion':  0.8,
}
mut_severity = mutation_classified.map(lambda x: severity_map[x]).values.astype(np.float32)

# 3. Tumor mutational burden (TMB)
tmb = mutation_binary.sum(axis=1).values.astype(np.float32).reshape(-1, 1)

# Combine all three
mut_combined = np.hstack([mut_binary_vals, mut_severity, tmb])

print(f"Mutation combined shape: {mut_combined.shape}")
print(f"  Binary: {mut_binary_vals.shape[1]}")
print(f"  Severity: {mut_severity.shape[1]}")
print(f"  TMB: {tmb.shape[1]}")

Mutation combined shape: (1904, 347)
  Binary: 173
  Severity: 173
  TMB: 1


In [6]:
# Process expression data
# Assume expression_data has patient ID as index or first column
# Align with clinical data
# X_expression = expression_data.set_index(expression_data.iloc[:, 0]).iloc[:, 1:]
X_expression = expression_data
X_expression = X_expression.loc[df.index]

# Scale expression features
scaler_expression = StandardScaler()
X_expression_scaled = scaler_expression.fit_transform(X_expression)

print(f"Expression features: {X_expression_scaled.shape}")

Expression features: (1904, 489)


# 5-Fold Cross-Validation Setup with Dual Encoder Architecture

Data is split into:
- **Encoder 1 (Clinical+Mutation)**: Clinical features + mutation features (binary + severity + TMB)
- **Encoder 2 (Expression)**: Gene expression data

In [7]:
class DualEncoderPreprocessor:
    """
    Handles preprocessing for dual-encoder architecture:
    - Encoder 1: Clinical + Mutation data
    - Encoder 2: Expression data
    """
    def __init__(self):
        self.scaler_clin_mut = StandardScaler()
        self.scaler_expr = StandardScaler()
        self.is_fitted = False
    
    def fit_transform(self, X_clin_mut, X_expr):
        """Fit scalers on training data and transform"""
        X_clin_mut_scaled = self.scaler_clin_mut.fit_transform(X_clin_mut)
        X_expr_scaled = self.scaler_expr.fit_transform(X_expr)
        self.is_fitted = True
        return X_clin_mut_scaled, X_expr_scaled
    
    def transform(self, X_clin_mut, X_expr):
        """Transform validation/test data using fitted scalers"""
        if not self.is_fitted:
            raise ValueError("Preprocessor must be fit first")
        X_clin_mut_scaled = self.scaler_clin_mut.transform(X_clin_mut)
        X_expr_scaled = self.scaler_expr.transform(X_expr)
        return X_clin_mut_scaled, X_expr_scaled

# Prepare combined clinical + mutation data
X_clin_mut = np.hstack([X_clinical_encoded.values, mut_combined])

print(f"Clinical+Mutation features: {X_clin_mut.shape}")
print(f"Expression features: {X_expression_scaled.shape}")
print(f"Survival outcomes: time={y_time.shape}, event={y_event.shape}")

Clinical+Mutation features: (1904, 376)
Expression features: (1904, 489)
Survival outcomes: time=(1904,), event=(1904,)


## Setting up Data for Cross Validation

In [8]:
def check(name, x):
    """Check statistics of processed data x"""
    print("\n", name)
    print("shape:", x.shape)
    print("nan:", np.isnan(x).sum())
    print("inf:", np.isinf(x).sum())
    print("min:", np.nanmin(x))
    print("max:", np.nanmax(x))

In [9]:
# Set up 5-fold cross-validation
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_fold_results = []

for fold_idx, (train_idx, test_idx) in enumerate(
    kfold.split(X_clin_mut, y_event)
):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx + 1}/5")
    print(f"{'='*70}")

    X_clin_mut_train, X_clin_mut_test = (
        X_clin_mut[train_idx],
        X_clin_mut[test_idx]
    )

    X_expr_train, X_expr_test = (
        X_expression_scaled[train_idx],
        X_expression_scaled[test_idx]
    )

    y_time_train, y_time_test = (
        y_time[train_idx],
        y_time[test_idx]
    )

    y_event_train, y_event_test = (
        y_event[train_idx],
        y_event[test_idx]
    )

    print("Train events:", y_event_train.sum())
    print("Test events:", y_event_test.sum())

    print(
        f"Train event rate: {y_event_train.mean():.3f}"
    )
    print(
        f"Test event rate: {y_event_test.mean():.3f}"
    )
    
    # Scale training and test data independently for this fold
    preprocessor = DualEncoderPreprocessor()
    X_clin_mut_train_scaled, X_expr_train_scaled = preprocessor.fit_transform(
        X_clin_mut_train, X_expr_train
    )
    X_clin_mut_test_scaled, X_expr_test_scaled = preprocessor.transform(
        X_clin_mut_test, X_expr_test
    )
    
    # Check data
    check("X_clin_mut_train_scaled", X_clin_mut_train_scaled)

    # Convert to tensors
    X_clin_mut_train_tensor = torch.tensor(X_clin_mut_train_scaled, dtype=torch.float32)
    X_expr_train_tensor = torch.tensor(X_expr_train_scaled, dtype=torch.float32)
    y_time_train_tensor = torch.tensor(y_time_train, dtype=torch.float32)
    y_event_train_tensor = torch.tensor(y_event_train, dtype=torch.float32)
    
    X_clin_mut_test_tensor = torch.tensor(X_clin_mut_test_scaled, dtype=torch.float32)
    X_expr_test_tensor = torch.tensor(X_expr_test_scaled, dtype=torch.float32)
    
    print(f"Clinical+Mutation train: {X_clin_mut_train_scaled.shape}")
    print(f"Expression train: {X_expr_train_scaled.shape}")
    print(f"Clinical+Mutation test: {X_clin_mut_test_scaled.shape}")
    print(f"Expression test: {X_expr_test_scaled.shape}")
    
    # Store fold data for use in training cells
    # (Training code will be in subsequent cells)


FOLD 1/5
Train events: 498
Test events: 124
Train event rate: 0.327
Test event rate: 0.325

 X_clin_mut_train_scaled
shape: (1523, 376)
nan: 107
inf: 0
min: -3.0167401148993704
max: 39.01281840626231
Clinical+Mutation train: (1523, 376)
Expression train: (1523, 489)
Clinical+Mutation test: (381, 376)
Expression test: (381, 489)

FOLD 2/5
Train events: 498
Test events: 124
Train event rate: 0.327
Test event rate: 0.325

 X_clin_mut_train_scaled
shape: (1523, 376)
nan: 109
inf: 0
min: -2.9943764690583916
max: 39.01281840626231
Clinical+Mutation train: (1523, 376)
Expression train: (1523, 489)
Clinical+Mutation test: (381, 376)
Expression test: (381, 489)

FOLD 3/5
Train events: 497
Test events: 125
Train event rate: 0.326
Test event rate: 0.328

 X_clin_mut_train_scaled
shape: (1523, 376)
nan: 109
inf: 0
min: -2.6972638072295663
max: 39.01281840626198
Clinical+Mutation train: (1523, 376)
Expression train: (1523, 489)
Clinical+Mutation test: (381, 376)
Expression test: (381, 489)

FOLD 4

# Loss Functions

In [10]:
def cox_loss(risk_scores, times, events):
    """
    Negative log partial likelihood for Cox proportional hazards.
    
    Args:
        risk_scores: Predicted risk scores (batch_size,)
        times: Survival times (batch_size,)
        events: Event indicators (batch_size,)
    
    Returns:
        Cox loss (scalar)
    """
    # Sort by time
    sorted_indices = torch.argsort(times, descending=False)
    sorted_risks = risk_scores[sorted_indices]
    sorted_events = events[sorted_indices]
    
    # Compute cumulative hazard
    cumsum_risks = torch.cumsum(torch.exp(sorted_risks), dim=0)
    
    # Negative log partial likelihood
    loss = 0.0
    for i in range(len(sorted_events)):
        if sorted_events[i] == 1:
            loss += -sorted_risks[i] + torch.log(cumsum_risks[i])
    
    return loss / torch.sum(sorted_events)

def compute_cindex(model, times, events, X_clin_mut, X_expr):
    """Compute concordance index for model evaluation"""
    model.eval()
    with torch.no_grad():
        risk_scores = model(X_clin_mut, X_expr).numpy()
    c_index = concordance_index(times, -risk_scores, events)
    return c_index

In [11]:
def cox_loss(risk_scores, times, events):
    """
    Negative log partial likelihood for Cox proportional hazards.
    
    Args:
        risk_scores: Predicted risk scores (batch_size,)
        times: Survival times (batch_size,)
        events: Event indicators (batch_size,)
    
    Returns:
        Cox loss (scalar)
    """
    # sort descending
    order = torch.argsort(times, descending=True)

    risk_scores = risk_scores[order]
    events = events[order]

    # numerically stable log(sum(exp()))
    log_cumsum = torch.logcumsumexp(risk_scores, dim=0)

    loss = -torch.sum((risk_scores - log_cumsum) * events)

    n_events = events.sum()

    if n_events == 0:
        return torch.tensor(
            0.0,
            device=risk_scores.device,
            requires_grad=True
        )

    return loss / n_events

def compute_cindex(model, times, events, X_clin_mut, X_expr):
    """Compute concordance index for model evaluation"""
    model.eval()
    with torch.no_grad():
        risk_scores = model(X_clin_mut, X_expr).numpy()
    c_index = concordance_index(times, -risk_scores, events)
    return c_index

# Model Architectures

In [12]:
class SurvivalAwareAutoencoder(nn.Module):
    """
    Survival-aware autoencoder for expression data.
    Encoder output is used as latent representation for downstream models.
    """
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

        # Survival head attached to latent space
        # Guides encoder to preserve survival-relevant variation
        self.survival_head = nn.Linear(latent_dim, 1)

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        risk = self.survival_head(z).squeeze(-1)
        return x_recon, z, risk

In [ ]:
class DualEncoderSurvivalNet(nn.Module):
    """
    Dual-encoder architecture combining clinical+mutation and expression data.
    
    Architecture:
    - Encoder 1: Clinical + Mutation → latent_dim_clin features
    - Encoder 2: Expression via pretrained autoencoder → latent_dim_expr features
    - Fusion: Concatenate latent representations → Survival head
    """
    def __init__(self, clin_mut_dim, expr_autoencoder, latent_dim_clin=32, latent_dim_expr=64):
        super().__init__()

        # Encoder 1: Clinical + Mutation
        self.clin_mut_enc = nn.Sequential(
            nn.Linear(clin_mut_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, latent_dim_clin),
            nn.ReLU()
        )

        # Encoder 2: Expression (pretrained autoencoder)
        self.expr_enc = expr_autoencoder

        # Fusion layer: Concatenate both latent spaces
        fusion_dim = latent_dim_clin + latent_dim_expr
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x_clin_mut, x_expr):
        """Forward pass through dual encoders and fusion"""
        z_clin_mut = self.clin_mut_enc(x_clin_mut)
        z_expr = self.expr_enc(x_expr)  # Uses encoder part of autoencoder
        z_fused = torch.cat([z_clin_mut, z_expr], dim=1)
        risk = self.fusion(z_fused).squeeze(-1)
        return risk
    
    # def forward(self, x_clin, x_expr):
    #     print("INPUTS:", torch.isnan(x_expr).any(), torch.isnan(x_clin).any())

    #     z_expr = self.expr_enc(x_expr)
    #     print("AFTER expr_enc:", torch.isnan(z_expr).any())

    #     z_clin = self.clin_mut_enc(x_clin)
    #     print("AFTER clin_enc:", torch.isnan(z_clin).any())

    #     z = torch.cat([z_clin, z_expr], dim=1)
    #     print("AFTER concat:", torch.isnan(z).any())

    #     out = self.fusion(z)
    #     print("AFTER fusion:", torch.isnan(out).any())

    #     return out

# Training Loop: Loss-Based Early Stopping (No Validation Set)

Key change: Early stopping is triggered by training loss plateau, NOT validation performance.
This prevents overfitting to validation data and ensures the model generalizes based on actual learning curve.

In [14]:
def train_survival_autoencoder(X_expr_train, y_time_train, y_event_train, latent_dim=64, epochs=500):
    """
    Pretrain survival-aware autoencoder.
    
    Early stopping: Based on training loss plateau (no validation set).
    """
    set_seed(42)
    
    expr_dim = X_expr_train.shape[1]
    ae = SurvivalAwareAutoencoder(input_dim=expr_dim, latent_dim=latent_dim)
    optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3, weight_decay=1e-4)
    recon_fn = nn.MSELoss()
    
    # Compute initial loss scales to balance reconstruction vs survival
    ae.eval()
    with torch.no_grad():
        recon_test, _, risk_test = ae(X_expr_train[:100])  # Sample for speed
        r_loss = recon_fn(recon_test, X_expr_train[:100]).item()
        s_loss = cox_loss(risk_test, y_time_train[:100], y_event_train[:100]).item()
    
    surv_weight = 0.05 * (r_loss / s_loss) if s_loss > 0 else 0.0
    print(f"Initial recon loss: {r_loss:.4f}")
    print(f"Initial surv loss: {s_loss:.4f}")
    print(f"Survival weight: {surv_weight:.4f}\n")
    
    # Early stopping: track training loss
    best_train_loss = float('inf')
    epochs_no_improve = 0
    patience = 30
    best_weights = None
    
    for epoch in range(epochs):
        ae.train()
        optimizer.zero_grad()
        
        recon, z, risk = ae(X_expr_train)
        recon_loss = recon_fn(recon, X_expr_train)
        surv_loss = cox_loss(risk, y_time_train, y_event_train)
        
        # Warmup phase: gradually introduce survival loss
        if epoch < 100:
            effective_weight = 0.0
        elif epoch < 200:
            effective_weight = surv_weight * (epoch - 100) / 100
        else:
            effective_weight = surv_weight
        
        total_loss = recon_loss + effective_weight * surv_loss
        total_loss.backward()
        optimizer.step()
        
        # Early stopping based on training loss improvement
        if epoch % 10 == 0:
            train_loss = total_loss.item()
            
            if train_loss < best_train_loss:
                best_train_loss = train_loss
                best_weights = {k: v.clone() for k, v in ae.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (training loss plateau)")
                break
            
            if epoch % 100 == 0:
                print(f"Epoch {epoch:>4} | Train loss: {train_loss:.4f} | "
                      f"Recon: {recon_loss.item():.4f} | Surv: {surv_loss.item():.4f}")
    
    ae.load_state_dict(best_weights)
    print(f"\nBest training loss: {best_train_loss:.4f}\n")
    return ae

In [49]:
def train_dual_encoder_survival_net(X_clin_mut_train, X_expr_train, y_time_train, y_event_train,
                                   expr_autoencoder, epochs_phase1=500, epochs_phase2=2000):
    """
    Two-phase training of dual-encoder survival network.
    
    Phase 1: Frozen expression encoder
    Phase 2: Fine-tune all layers
    
    Early stopping: Based on training loss plateau (no validation set).
    """
    
    # Phase 1: Frozen encoder
    print("\n" + "="*70)
    print("Phase 1: Training with frozen expression encoder")
    print("="*70)
    
    model = DualEncoderSurvivalNet(
        clin_mut_dim=X_clin_mut_train.shape[1],
        expr_autoencoder=expr_autoencoder,
        latent_dim_clin=32,
        latent_dim_expr=64
    )
    
    # Freeze expression encoder
    for param in model.expr_enc.parameters():
        param.requires_grad = False
    
    optimizer_p1 = torch.optim.Adam([
        {"params": model.clin_mut_enc.parameters(), "lr": 1e-3},
        {"params": model.fusion.parameters(), "lr": 1e-3}
    ], weight_decay=1e-3)
    
    best_train_loss = float('inf')
    epochs_no_improve = 0
    patience = 50
    # best_weights = None
    # Updated 
    best_weights = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs_phase1):
        model.train()
        optimizer_p1.zero_grad()
        
        risk = model(X_clin_mut_train, X_expr_train)

        print("Risk stats:")
        print("min:", risk.min().item())
        print("max:", risk.max().item())
        print("mean:", risk.mean().item())
        print("contains nan:", torch.isnan(risk).any().item())
        print("contains inf:", torch.isinf(risk).any().item())

        loss = cox_loss(risk, y_time_train, y_event_train)

        print("Loss:", loss.item())
        
        loss.backward()
        optimizer_p1.step()
        
        # Early stopping based on training loss
        if epoch % 10 == 0:
            train_loss = loss.item()
            
            if train_loss < best_train_loss:
                best_train_loss = train_loss
                best_weights = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch} (training loss plateau)")
                break
            
            if epoch % 100 == 0:
                print(f"  Epoch {epoch:>4} | Train loss: {train_loss:.4f}")
    
    model.load_state_dict(best_weights)
    print(f"Phase 1 best training loss: {best_train_loss:.4f}\n")
    
    # Phase 2: Unfreeze and fine-tune
    print("="*70)
    print("Phase 2: Fine-tuning all layers")
    print("="*70)
    
    for param in model.expr_enc.parameters():
        param.requires_grad = True
    
    optimizer_p2 = torch.optim.Adam([
        {"params": model.clin_mut_enc.parameters(), "lr": 1e-4},
        {"params": model.expr_enc.parameters(), "lr": 1e-5},
        {"params": model.fusion.parameters(), "lr": 1e-4}
    ], weight_decay=1e-3)
    
    best_train_loss_p2 = best_train_loss
    epochs_no_improve = 0
    patience = 100
    
    for epoch in range(epochs_phase2):
        model.train()
        optimizer_p2.zero_grad()
        
        risk = model(X_clin_mut_train, X_expr_train)
        loss = cox_loss(risk, y_time_train, y_event_train)
        loss.backward()
        optimizer_p2.step()
        
        # Early stopping based on training loss
        if epoch % 10 == 0:
            train_loss = loss.item()
            
            if train_loss < best_train_loss_p2:
                best_train_loss_p2 = train_loss
                best_weights = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch} (training loss plateau)")
                break
            
            if epoch % 100 == 0:
                print(f"  Epoch {epoch:>4} | Train loss: {train_loss:.4f}")
    
    model.load_state_dict(best_weights)
    print(f"Phase 2 best training loss: {best_train_loss_p2:.4f}\n")
    
    return model

# Execute 5-Fold Cross-Validation

In [15]:
# Set up 5-fold cross-validation
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_fold_results = []

for fold_idx, (train_idx, test_idx) in enumerate(
    kfold.split(X_clin_mut, y_event)
):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx + 1}/5")
    print(f"{'='*70}")

    X_clin_mut_train, X_clin_mut_test = (
        X_clin_mut[train_idx],
        X_clin_mut[test_idx]
    )

    X_expr_train, X_expr_test = (
        X_expression_scaled[train_idx],
        X_expression_scaled[test_idx]
    )

    y_time_train, y_time_test = (
        y_time[train_idx],
        y_time[test_idx]
    )

    y_event_train, y_event_test = (
        y_event[train_idx],
        y_event[test_idx]
    )

    print("Train events:", y_event_train.sum())
    print("Test events:", y_event_test.sum())

    print(
        f"Train event rate: {y_event_train.mean():.3f}"
    )
    print(
        f"Test event rate: {y_event_test.mean():.3f}"
    )
    
    # Preprocess for this fold
    preprocessor = DualEncoderPreprocessor()
    X_clin_mut_train_scaled, X_expr_train_scaled = preprocessor.fit_transform(
        X_clin_mut_train, X_expr_train
    )
    X_clin_mut_test_scaled, X_expr_test_scaled = preprocessor.transform(
        X_clin_mut_test, X_expr_test
    )
    
    # Convert to tensors
    X_clin_mut_train_tensor = torch.tensor(X_clin_mut_train_scaled, dtype=torch.float32)
    X_expr_train_tensor = torch.tensor(X_expr_train_scaled, dtype=torch.float32)
    y_time_train_tensor = torch.tensor(y_time_train, dtype=torch.float32)
    y_event_train_tensor = torch.tensor(y_event_train, dtype=torch.float32)
    
    X_clin_mut_test_tensor = torch.tensor(X_clin_mut_test_scaled, dtype=torch.float32)
    X_expr_test_tensor = torch.tensor(X_expr_test_scaled, dtype=torch.float32)
    
    print(f"Train: {X_clin_mut_train_tensor.shape[0]} samples")
    print(f"Test: {X_clin_mut_test_tensor.shape[0]} samples")
    print(f"Clinical+Mutation features: {X_clin_mut_train_tensor.shape[1]}")
    print(f"Expression features: {X_expr_train_tensor.shape[1]}")
    
    # Step 1: Pretrain survival-aware autoencoder
    print(f"\nStep 1: Pretraining Survival-Aware Autoencoder")
    ae = train_survival_autoencoder(
        X_expr_train_tensor, y_time_train_tensor, y_event_train_tensor,
        latent_dim=64
    )
    
    # Step 2: Train dual-encoder survival network
    print(f"\nStep 2: Training Dual-Encoder Survival Network")
    model = train_dual_encoder_survival_net(
        X_clin_mut_train_tensor, X_expr_train_tensor,
        y_time_train_tensor, y_event_train_tensor,
        expr_autoencoder=ae.encoder
    )
    
    # Step 3: Evaluate on test set
    print(f"\nStep 3: Evaluating on Test Set")
    test_cindex = compute_cindex(
        model, y_time_test, y_event_test,
        X_clin_mut_test_tensor, X_expr_test_tensor
    )
    print(f"Test C-index: {test_cindex:.4f}")
    
    cv_fold_results.append({
        'fold': fold_idx + 1,
        'test_cindex': test_cindex
    })


FOLD 1/5
Train events: 498
Test events: 124
Train event rate: 0.327
Test event rate: 0.325
Train: 1523 samples
Test: 381 samples
Clinical+Mutation features: 376
Expression features: 489

Step 1: Pretraining Survival-Aware Autoencoder
Initial recon loss: 1.1409
Initial surv loss: 4.1255
Survival weight: 0.0138

Epoch    0 | Train loss: 1.0017 | Recon: 1.0017 | Surv: 6.8198
Epoch  100 | Train loss: 0.6750 | Recon: 0.6750 | Surv: 6.8251
Epoch  200 | Train loss: 0.7013 | Recon: 0.6098 | Surv: 6.6128
Epoch  300 | Train loss: 0.6641 | Recon: 0.5796 | Surv: 6.1097
Epoch  400 | Train loss: 0.6310 | Recon: 0.5598 | Surv: 5.1463

Best training loss: 0.6170


Step 2: Training Dual-Encoder Survival Network


NameError: name 'train_dual_encoder_survival_net' is not defined

# Cross-Validation Summary

In [ ]:
# Summarize CV results
cv_results_df = pd.DataFrame(cv_fold_results)

print(f"\n{'='*70}")
print("5-FOLD CROSS-VALIDATION RESULTS")
print(f"{'='*70}")
print(cv_results_df.to_string(index=False))
print(f"{'='*70}")
print(f"Mean C-index: {cv_results_df['test_cindex'].mean():.4f} ± {cv_results_df['test_cindex'].std():.4f}")
print(f"Min C-index:  {cv_results_df['test_cindex'].min():.4f}")
print(f"Max C-index:  {cv_results_df['test_cindex'].max():.4f}")
print(f"{'='*70}")

In [ ]:
# Summary of improvements
print("""
SUMMARY OF IMPROVEMENTS
═══════════════════════════════════════════════════════════════════════

1. CLEAN DATA PREPROCESSING
   ✓ Separate preprocessing for dual encoders:
     - Encoder 1: Clinical features + combined mutation features
       (binary mutations + severity-weighted + TMB)
     - Encoder 2: Gene expression data
   ✓ DualEncoderPreprocessor class handles fit/transform independently
   ✓ Prevents data leakage by fitting scalers on train only

2. 5-FOLD CROSS-VALIDATION
   ✓ KFold splits training data into 5 folds
   ✓ Each fold has independent preprocessing (no leakage)
   ✓ Models trained on fold-specific training data
   ✓ Evaluation on held-out test folds
   ✓ Results aggregated: mean ± std C-index across folds

3. LOSS-BASED EARLY STOPPING (NO VALIDATION SET)
   ✓ Early stopping triggered by training loss plateau
   ✓ NO validation set is used for early stopping decisions
   ✓ Prevents overfitting to validation data
   ✓ Both autoencoder and survival network use this approach:
     - Pretraining: tracks best training loss
     - Phase 1: frozen encoder, tracks training loss
     - Phase 2: fine-tuning, tracks training loss
   ✓ Patience parameter (30-100 epochs) controls stopping threshold

4. DUAL ENCODER ARCHITECTURE
   ✓ Clinical+Mutation Encoder: 128→64→32 dims
   ✓ Expression Encoder: from pretrained survival-aware autoencoder
   ✓ Fusion Layer: concatenates both latent spaces → survival head
   ✓ Two-phase training:
     - Phase 1: learn clinical representation with frozen expression encoder
     - Phase 2: fine-tune all parameters with lower learning rates

═══════════════════════════════════════════════════════════════════════
""")